# Minimal Token Sequence Test

Use this notebook to run tokens one by one, edit the sequence, and inspect the `State` after each step.

This version uses **kernels_playground** utilities for robust data loading.

## Colab Setup

Clone both repos and install dependencies.

In [ ]:
import os, sys

# ── Clone graph_Time_series (the token framework) ──
%cd /content
if not os.path.exists("graph_Time_series"):
    !git clone https://github.com/chahineNejm/graph_Time_series

# ── Clone kernels_playground (data & decomposition utilities) ──
if not os.path.exists("kernels_playground"):
    !git clone https://github.com/chahineNejm/kernels_playground

# ── Make both importable ──
for p in ["/content/graph_Time_series", "/content/kernels_playground", "/content/kernels_playground/first_tests"]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Install minimal deps ──
!pip install -q datasets

print("cwd:", os.getcwd())
print("graph_Time_series repo:", os.path.isdir("/content/graph_Time_series/graph_Time_series"))
print("kernels_playground repo:", os.path.isdir("/content/kernels_playground/first_tests/utils"))

## Load Current Token Files

Load the graph_Time_series tokens without importing the full package.

In [ ]:
from pathlib import Path
import importlib.util
import sys
import types

import numpy as np


# ── Bootstrap graph_Time_series modules manually ──
PACKAGE_DIR = Path("/content/graph_Time_series/graph_Time_series")
assert PACKAGE_DIR.exists(), f"Package dir not found: {PACKAGE_DIR}"


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


pkg = types.ModuleType("graph_Time_series")
pkg.__path__ = [str(PACKAGE_DIR)]
sys.modules.setdefault("graph_Time_series", pkg)

blocks = types.ModuleType("graph_Time_series.token_blocks")
blocks.__path__ = [str(PACKAGE_DIR / "token_blocks")]
sys.modules.setdefault("graph_Time_series.token_blocks", blocks)

state_mod = load_module("graph_Time_series.state", PACKAGE_DIR / "state.py")
token_mod = load_module("graph_Time_series.token", PACKAGE_DIR / "token.py")
norm_mod = load_module(
    "graph_Time_series.token_blocks.normalization",
    PACKAGE_DIR / "token_blocks" / "normalization.py",
)
rbf_mod = load_module(
    "graph_Time_series.token_blocks.kernel_rbf",
    PACKAGE_DIR / "token_blocks" / "kernel_rbf.py",
)

State = state_mod.State
ZNormalizationToken = norm_mod.ZNormalizationToken
KernelRBFToken = rbf_mod.KernelRBFToken

print("Loaded token modules from", PACKAGE_DIR)

## Choose Tokens

In [ ]:
TOKENS = {
    "ZNormalization": ZNormalizationToken(),
    "kernel_rbf": KernelRBFToken(),
}

# Edit this list to test your manual chain.
TOKEN_SEQUENCE = [
    "ZNormalization",
    "kernel_rbf",
]

## Load Electricity Data

Uses `kernels_playground/first_tests/utils/data.py` for robust data loading from GiftEvalParquet.

In [ ]:
from utils.config import DEFAULT_CONFIG, DATASETS
from utils.data import build_examples, extract_history_future, clean_series
from utils.augmentation import uniform_length

MAX_RUN_SAMPLES = 24
HOLDOUT_SAMPLES = 8

# ── 1. Load raw examples (keeps original lengths) ──
raw = build_examples(
    config="electricity_H_long",
    start=0,
    stop=100,
    step=1,
    dataset_name=DATASETS["eval"],
)
print(f"Loaded {len(raw)} raw examples")

# ── 2. Check length distribution ──
hist_lens = [len(e["history"]) for e in raw]
fut_lens  = [len(e["future"]) for e in raw]
print(f"History lengths: {min(hist_lens)} – {max(hist_lens)}")
print(f"Future lengths:  {min(fut_lens)} – {max(fut_lens)}")

# ── 3. Stack into 2D arrays for State (n_samples, timesteps) ──
needed = MAX_RUN_SAMPLES + HOLDOUT_SAMPLES
examples = raw[:needed]

min_hist = min(len(e["history"]) for e in examples)
min_fut  = min(len(e["future"]) for e in examples)

H_all = np.stack([e["history"][-min_hist:] for e in examples]).astype(np.float32)
F_all = np.stack([e["future"][:min_fut] for e in examples]).astype(np.float32)

H = H_all[:MAX_RUN_SAMPLES]
F = F_all[:MAX_RUN_SAMPLES]
H_holdout = H_all[MAX_RUN_SAMPLES:]
F_holdout = F_all[MAX_RUN_SAMPLES:]

print(f"\nRun data:     H={H.shape}, F={F.shape}")
print(f"Held-out data: H={H_holdout.shape}, F={F_holdout.shape}")

## Run Sequence

In [ ]:
def feature_shapes(values):
    return {k: getattr(v, "shape", None) for k, v in values.items()}


def summarize_state(state, label):
    print(f"\n--- {label} ---")
    print(state)
    print("token_sequence:", state.token_sequence)
    print("class_counts:", state.class_counts)
    print("historical_features:", feature_shapes(state.historical_features))
    print("future_features:", feature_shapes(state.future_features))
    print("flags:", state.flags)
    print("transforms:", [t.name for t in state.transform_stack])
    print("prediction_names:", state.prediction_names)
    print("active_target_base:", state.active_target_base.shape)
    print("current_target:", state.current_target.shape)


def run_sequence(sequence, H, F):
    state = State(H, F)
    summarize_state(state, "after init")

    for name in sequence:
        token = TOKENS[name]
        print(f"\nToken: {name}")
        can_apply = token.can_apply(state)
        print("can_apply:", can_apply)
        if not can_apply:
            raise RuntimeError(f"Token {name} cannot apply to current state")
        state = token.apply(state)
        summarize_state(state, f"after {name}")

    forecast = state.get_final_prediction()
    print("\nFinal forecast shape:", forecast.shape)
    print("Final forecast sample:", np.round(forecast[0], 3))
    return state, forecast


state, forecast = run_sequence(TOKEN_SEQUENCE, H, F)

## Inspect State

In [ ]:
print("token_sequence:", state.token_sequence)
print("class_counts:", state.class_counts)
print("historical_features:", list(state.historical_features.keys()))
print("future_features:", list(state.future_features.keys()))
print("flags:", state.flags)
print("transforms:", [t.name for t in state.transform_stack])
print("prediction_names:", state.prediction_names)

In [ ]:
state.print_log()